# PrivateLocalAgent

**Kernel → Restart Kernel**，然后只运行下面 **一个** 代码单元格。

In [ ]:
import os, sys, subprocess, re, shutil, socket
from pathlib import Path
from IPython.display import HTML, display, clear_output

def log(msg):
    print(msg, flush=True)

ROOT = Path("/workspace/Radeon-hackathon-2026-07")
if not (ROOT / "src" / "config.py").is_file():
    here = Path.cwd()
    ROOT = here.parent if here.name == "notebooks" else here
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
log(f"ROOT={ROOT}")

persist = Path("/workspace/persistence")
if not persist.is_dir():
    persist = Path("/persistent")
if persist.is_dir():
    os.environ.setdefault("PLA_DATA_ROOT", str(persist / "PrivateLocalAgent"))
    os.environ.setdefault("HF_HOME", str(persist / "huggingface"))
    Path(os.environ["PLA_DATA_ROOT"]).mkdir(parents=True, exist_ok=True)
    Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("USE_ROCM_AITER_ROPE_BACKEND", "0")
os.environ["PLA_NOTEBOOK_UI_PORT"] = "7900"
os.environ["PLA_ALLOW_PUBLIC"] = "1"
os.environ["HTTP_HOST"] = "127.0.0.1"
os.environ["HTTP_PORT"] = "7900"
os.environ["PLA_OPEN_BROWSER"] = "0"

log("[0] pip...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "chromadb", "sentence-transformers", "pypdf", "pyyaml",
    "python-dotenv", "pydantic", "openai",
    "transformers", "accelerate", "safetensors", "sentencepiece",
    "Pillow", "rapidocr-onnxruntime",
])

from src.agent.agent import PrivateAgent
from src.agent.multi_agent import MultiAgentOrchestrator
from src.agent.tools import ToolRegistry
from src.apps.judge_script import ensure_judge_ocr_image
from src.config import load_settings
from src.llm.backend import build_llm
from src.memory.memory import SessionMemory
from src.privacy.audit import AuditTrail
from src.rag.store import VectorStore
from src.skills import SkillRegistry
from src.app import notebook_visual as nv

settings = load_settings()
upload_dir = settings.resolve(settings.paths.upload_dir)
ensure_judge_ocr_image(upload_dir)

log("[1] KB...")
store = VectorStore(settings)
n = store.ensure_sample_docs(settings.resolve(settings.paths.sample_docs))
log(f"    ingested={n} chunks={store.count()}")

memory = SessionMemory(settings.resolve(settings.agent.memory_path))
skills = SkillRegistry(settings.resolve(settings.paths.generated_projects))
tools = ToolRegistry(store, memory, upload_dir, skill_registry=skills)
audit = AuditTrail(settings.resolve("data/memory/audit.jsonl"))

log("[2] load LLM...")
llm = build_llm(settings.llm)
agent = PrivateAgent(llm, tools, memory, settings.agent.max_steps, audit=audit)
orch = MultiAgentOrchestrator(agent, tools)
log("ready")

PORT = 7900
local_url = f"http://127.0.0.1:{PORT}/"
log("[3] web UI...")
try:
    nv._start_web_for_orch(orch, default_mode="chat", port=PORT)
except OSError:
    log("    port busy — reuse existing :7900")

log("[4] rc-tunnel...")
installer = Path("/var/run/secrets/frp-self-service/install")
rc = shutil.which("rc-tunnel") or str(Path.home() / ".local/bin/rc-tunnel")
if installer.exists() and not Path(rc).exists():
    subprocess.run(["bash", str(installer)], check=False)
    rc = shutil.which("rc-tunnel") or str(Path.home() / ".local/bin/rc-tunnel")
os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ.get("PATH", "")

public_url = None
if Path(rc).exists():
    subprocess.run([rc, "stop"], check=False, capture_output=True)
    proc = subprocess.run([rc, "expose", "--port", str(PORT)], capture_output=True, text=True, timeout=120)
    out = (proc.stdout or "") + "\n" + (proc.stderr or "")
    log(out)
    m = re.search(r"https?://[^\s]*radeon\.firstdg\.ai[^\s]*", out)
    if m:
        public_url = m.group(0).rstrip("/")
        if public_url.startswith("http://"):
            public_url = "https://" + public_url[len("http://"):]
else:
    log("rc-tunnel missing: /var/run/secrets/frp-self-service/install")

# reuse previous URL if expose is in quarantine but tunnel already up
if not public_url:
    prev = Path("/tmp/pla_rc_tunnel.log")
    if prev.exists():
        m = re.search(r"https?://[^\s]*radeon\.firstdg\.ai[^\s]*", prev.read_text(errors="ignore"))
        if m:
            public_url = m.group(0).rstrip("/").replace("http://", "https://", 1)

embed = public_url or local_url
log(f"local={local_url}")
log(f"public={public_url}")
log(f"embed={embed}")

clear_output(wait=True)
display(HTML(f"""
<div style="font-family:Segoe UI,PingFang SC,Microsoft YaHei,sans-serif;color:#e5e7eb">
  <div style="padding:12px 14px;border-radius:12px;background:#0f172a;border:1px solid #334155;margin-bottom:10px">
    <b style="color:#99f6e4;font-size:20px">PrivateLocalAgent</b>
    <div style="margin-top:8px;font-size:13px;line-height:1.7">
      <a href="{embed}" target="_blank" rel="noopener" style="color:#5eead4">打开外部完整界面</a>
      &nbsp;|&nbsp; 本机 <code style="color:#fbbf24">{local_url}</code>
      {('&nbsp;|&nbsp; 公网 <code style="color:#fbbf24">'+public_url+'</code>') if public_url else ''}
    </div>
  </div>
  <iframe src="{embed}" title="PrivateLocalAgent"
    style="width:100%;height:780px;border:1px solid #334155;border-radius:14px;background:#0b1220"></iframe>
</div>
"""))

ui = type("UI", (), {"orch": orch, "local_url": local_url, "public_url": public_url, "port": PORT})()
print("done", embed, flush=True)